# Caching

There are multiple meanings of the word "caching" when it comes to computers and, particularly, code performance. In this section we will look at two common meanings relevant to code performance.

## Caching Variables

When certain values are calculated repeatedly, it may be worth saving their values rather than calculating them repeatedly. The more times the value is calculated and the more complex the calculation, the more viable this strategy becomes.

For example, consider the following codes which aim to calculate:

$\sum\limits_{i=0}^{1000}\sum\limits_{j=0}^{1000}\sin{\left(\frac{i\pi}{1000}\right)}\sin{\left(\frac{j\pi}{1000}\right)} $

In [ ]:
import math
%load_ext line_profiler

def sum_function():
  result=0

  for i in range(0, 1001):
    for j in range(0, 1001):
      result = result + math.sin(i * math.pi / 1000)*math.sin(j * math.pi / 1000)

  return result

%lprun -f sum_function print(sum_function())

The first thing we might notice is that we're currently performing the operation $\frac{\pi}{1000}$ 1,000,000 times and this will always have the same value. We can pre-calculate this value once and use it repeatedly:

In [ ]:
import math
%load_ext line_profiler

def sum_function():
  result=0
  pi_over_1000 = math.pi / 1000

  for i in range(0, 1001):
    for j in range(0, 1001):
      result = result + math.sin(i * pi_over_1000)*math.sin(j * pi_over_1000)

  return result

%lprun -f sum_function print(sum_function())

The next thing we might notice is that there are two nested ```for``` loops. The variable ```j``` takes 1000 different values for each value ```i``` takes. Thus, we can calculate the value $\sin{\left(\frac{i\pi}{1000}\right)}$ and cache it inside the outer loop:

In [ ]:
import math
%load_ext line_profiler

def sum_function():
  result=0
  pi_over_1000=math.pi/1000

  for i in range(0, 1001):
    sin_i = math.sin(i * pi_over_1000)
    for j in range(0, 1001):
      result = result + sin_i * math.sin(j * pi_over_1000)

  return result

%lprun -f sum_function print(sum_function())

This reduces the number of times we call ```math.sin``` from 2,000,000 to 1,001,000.

Finally, we might notice that we actually only call ```math.sin``` with 1,000 different values so we can actually create a list of the resultant values to cache them:

In [ ]:
import math
%load_ext line_profiler

def sum_function():
  result=0
  pi_over_1000=math.pi/1000

  sin_values=[]

  for i in range(0, 1001):
    sin_values.append(math.sin(i * pi_over_1000))

  for sin_i in sin_values:
    for sin_j in sin_values:
      result = result + sin_i * sin_j

  return result

%lprun -f sum_function print(sum_function())

The resulting code calls the ```sin``` function 1,000 times and runs in about half the time compared to the original code. However, it does use more memory and is less readable.

### Exercise: Average Temperature

For a given geographical region, it is known that the temperaure of a location can reliably be defined by the x and y coordinates $x$ and $y$ and the time measured in days through the year $t$ according to the following formula:

$$
T(x, y, t) = 0.01x + 0.005y + 10\sin\left(\frac{t}{365}\right) + 20
$$

The code below calculates the average temperature of the region at a particular time in the year `t` by dividing the region into a series of rectangular bins, then taking the temperature at the centre of each of them, summing these values, then dividing by the number of bins. There are two versions of the code. The first is the initial version, which you should not edit. The second is intended for you to edit and apply some caching techniques to make it run faster. What repeated calculations can you cache the result of? Consider which terms are constant, which are only functions of `i` and which are only functions of `j`.

In [ ]:
# Original version: DO NOT EDIT
import math
%load_ext line_profiler

x_extent = 100
y_extent = 50

x_bins = 1000
y_bins = 1000

def average_temperature(t):
    total_temp = 0
    for i in range(x_bins):
        for j in range(y_bins):
            x = (i + 0.5) * (x_extent / x_bins)
            y = (j + 0.5) * (y_extent / y_bins)
            total_temp = total_temp + 0.01 * x + 0.005 * y + 10 * math.sin(t / 365) + 20
    print("Average Temperature: ", total_temp / (x_bins * y_bins))

%lprun -f average_temperature average_temperature(100)

In [ ]:
# Version for you to edit
import math
%load_ext line_profiler

x_extent = 100
y_extent = 50

x_bins = 1000
y_bins = 1000

def average_temperature(t):
    total_temp = 0
    for i in range(x_bins):
        for j in range(y_bins):
            x = (i + 0.5) * (x_extent / x_bins)
            y = (j + 0.5) * (y_extent / y_bins)
            total_temp = total_temp + 0.01 * x + 0.005 * y + 10 * math.sin(t / 365) + 20
    print("Average Temperature: ", total_temp / (x_bins * y_bins))

%lprun -f average_temperature average_temperature(100)

## Caching Function Results

Often, functions will be called repeatedly with the same values passed as arguments and, thus, returning the same result. If a function is complex, it's possible to save a significant amount of time by noting a function has been called before and returning the value that was called then without performing the body of the function. For example, the following code tests a recursive function designed to calculate the Fibonacci sequence:

In [ ]:
import cProfile

def fibonacci(n):
  if n < 2:
    return n
  else:
    return fibonacci(n-1) + fibonacci(n-2)

cProfile.run('print(fibonacci(32))')

When we run this code we see the function is called a large number of times to calculate the desired value. We know that the function will only be called with values of ```n``` less than 32, however. This means if we could cache the results of the function with those 32 values of ```n``` we could eliminate the bodies of most of the functions and thus most of the function calls and most of the time spent.

It's possible to tell Python to cache the results of calls to a function automatically. This stores the results behind the scenes for the last few combinations of arguments used. To do this we may import the ```lru_cache``` "decorator" from the ```functools``` module and adding it to the function:

In [ ]:
import cProfile
from functools import lru_cache

@lru_cache(maxsize=32)
def fibonacci(n):
  if n < 2:
    return n
  else:
    return fibonacci(n-1) + fibonacci(n-2)

cProfile.run('print(fibonacci(32))')

Decorators, when added to functions, modify how the function behaves. In this case, the ```lru_cache``` decorator causes the results of the functions for the last ```maxsize``` unique combinations of arguments provided. When one of the stored combinations of arguments is used to call the function, the cached value is returned instead of calling the function in its entirety.

In this case, the body of most function is bypassed in almost every case and, as almost all function calls are in the bodies of function, most function calls are also eliminated. This means the number of function calls is reduced from over 7,000,000 to just 33 and the run-time is also decreased from over a second to almost nothing.

This example happens to be a case where this tactic is particularly effective as we can guarantee that there will only be a small number of values passed as an argument and the number of function calls was initially very high.

### Exercise: Morse Code

Morse Code is a way of encoding characters into a series of dots and dashes, with each combination representing a unique character. The code below contains functions which translate a single word (a part of a string separated by spaces) or a long messsage to Morse Code.

These functions are used to translate some navigational data to Morse Code. This navigational data is a series of pairs of data, with each pair comprised a direction ("North", "South", "East" or "West") and a distance (a multiple of 10 between 0 and 100 inclusive), separated by a space.

Another function generates a random message of a specified length. Finally, there is a function call to translate a message with 10,000 pairs of values to Morse Code, instrumented with cProfile.

There are two versions of the code. The original version which you should not edit, and a version for you to edit. Where should you put the `lru_cache` decorator to speed up the code? What is the minimum size of cache you should use? Run your edited code and compare it to the original code. How much faster is your version?

In [ ]:
# Original version: DO NOT EDIT
import cProfile
import random

def translate_word_to_morse(word):
    # This function translates a single word to morse code
    # Each letter is separated by a space
    morse_dict = {"a":".-", "b":"-...", "c":"-.-.", "d":"-..", "e":".", "f":"..-.", "g":"--.", "h":"....", "i":"..", "j":".---", "k":"-.-", "l":".-..", "m":"--", "n":"-.", "o":"---", "p":".--.", "q":"--.-", "r":".-.", "s":"...", "t":"-", "u":"..-", "v":"...-", "w":".--", "x":"-..-", "y":"-.--", "z":"--..", "1":".----", "2":"..---", "3":"...--", "4":"....-", "5":".....", "6":"-....", "7":"--...", "8":"---..", "9":"----.", "0":"-----"}
    morse_translation = ""
    for letter in word.lower():
        morse_translation += morse_dict[letter] + " "
    return morse_translation.strip()

def translate_message_to_morse(message):
    # This function translates a full message to morse code
    # Each word is separated by three spaces
    morse_message = ""
    for word in message.split(" "):
        morse_message += translate_word_to_morse(word) + "   "
    return morse_message.strip()

def generate_random_message(length):
    # This first generates a message with length pairs of directions and distances
    # The direction will always be one of "North", "South", "East" and "West"
    # The distance will always be a multiple of 10 between 0 and 100
    directions = ["North", "East", "South", "West"]
    message = ""
    for i in range(length):
        direction = random.choice(directions)
        distance = 10 * random.randint(0, 10)
        message += direction + " " + str(distance) + " "
    return message

# Sample calls to the functions
print("'sos' Translation: ", translate_word_to_morse("sos"))
print("'sos help' Translation: ", translate_message_to_morse("sos help"))
print("Length 5 message: ", generate_random_message(5))

message = generate_random_message(10000)

cProfile.run('translate_message_to_morse(message)')

In [ ]:
# Edit this version
import cProfile
import random

def translate_word_to_morse(word):
    # This function translates a single word to morse code
    # Each letter is separated by a space
    morse_dict = {"a":".-", "b":"-...", "c":"-.-.", "d":"-..", "e":".", "f":"..-.", "g":"--.", "h":"....", "i":"..", "j":".---", "k":"-.-", "l":".-..", "m":"--", "n":"-.", "o":"---", "p":".--.", "q":"--.-", "r":".-.", "s":"...", "t":"-", "u":"..-", "v":"...-", "w":".--", "x":"-..-", "y":"-.--", "z":"--..", "1":".----", "2":"..---", "3":"...--", "4":"....-", "5":".....", "6":"-....", "7":"--...", "8":"---..", "9":"----.", "0":"-----"}
    morse_translation = ""
    for letter in word.lower():
        morse_translation += morse_dict[letter] + " "
    return morse_translation.strip()

def translate_message_to_morse(message):
    # This function translates a full message to morse code
    # Each word is separated by three spaces
    morse_message = ""
    for word in message.split(" "):
        morse_message += translate_word_to_morse(word) + "   "
    return morse_message.strip()

def generate_random_message(length):
    # This first generates a message with length pairs of directions and distances
    # The direction will always be one of "North", "South", "East" and "West"
    # The distance will always be a multiple of 10 between 0 and 100
    directions = ["North", "East", "South", "West"]
    message = ""
    for i in range(length):
        direction = random.choice(directions)
        distance = 10 * random.randint(0, 10)
        message += direction + " " + str(distance) + " "
    return message

# Sample calls to the functions
print("'sos' Translation: ", translate_word_to_morse("sos"))
print("'sos help' Translation: ", translate_message_to_morse("sos help"))
print("Length 5 message: ", generate_random_message(5))

message = generate_random_message(10000)

cProfile.run('translate_message_to_morse(message)')